# Answer Questions

Query the index and feed the top N results to ChatGPT.

This notebook is the whole of retrieval-augmented generation, written out by hand.
There is no chain object hiding the steps, so you can see each one and poke at it:

1. **Embed** the question into a 1536-dimensional vector.
2. **Query** the Pinecone index for the passages whose vectors sit nearest to it.
3. **Build a prompt** out of those passages, as many as fit in a character budget.
4. **Ask the model** that prompt and read back the answer.
5. **Show the sources**, so the answer can be checked against the talks it came from.

Steps 1-4 are four small functions below, and together they are about forty lines.
This is the same sequence the live search server runs for every query
(`server/server/main.py`).

In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [ ]:
import os
import textwrap
from pathlib import Path

from dotenv import dotenv_values
from openai import OpenAI
from pinecone import Pinecone

In [ ]:
# The index holds 1536-dimensional text-embedding-ada-002 vectors. The question has
# to be embedded with that same model: a different model writes into a different
# vector space, so the "nearest" passages would come back as noise rather than an
# error. This is the one constant in the notebook you must not change.
text_embedding_model_name = "text-embedding-ada-002"

# The model that writes the answer, matching what the live server uses today.
text_generation_model_name = "gpt-5-mini"

# The production index is serverless, so it is addressed by host rather than by name.
index_host = "https://conf-ada-002-svrless-29e6444.svc.aped-4627-b74a.pinecone.io"

search_limit = 20  # passages to retrieve (the old RetrievalQA chain used k=20)
prompt_limit = 3000  # characters of the prompt to spend on retrieved context
max_answer_tokens = 500

## Initialize OpenAI and Pinecone

In [ ]:
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

### Read-only by construction

This index is **production** — it is what the live site searches. A notebook has no
business writing to it, and "remember not to call upsert" is not a safeguard. So the
raw index object never gets a name in this notebook: `connect_to_index` keeps it
inside a wrapper that exposes only the two read operations. `index.upsert(...)` or
`index.delete(...)` raises `AttributeError` here instead of reaching Pinecone.

The key matters too. `models/.env`'s `PINECONE_API_KEY` is a **development** key
pointing at an empty project — using it gives zero matches and no error at all. The
production key lives in the sibling server repo, and that is where this reads it
from.

In [ ]:
class ReadOnlyIndex:
    """A read-only view of a Pinecone index: it can search, and that is all."""

    def __init__(self, index):
        self._index = index

    def query(self, *, vector, top_k, include_metadata=True, **kwargs):
        """Return the top_k vectors nearest to the given vector."""
        return self._index.query(
            vector=vector, top_k=top_k, include_metadata=include_metadata, **kwargs
        )

    def describe_index_stats(self):
        """Return vector counts and dimensions for the index."""
        return self._index.describe_index_stats()


def get_pinecone_prod_key():
    """Resolve the production Pinecone key, or explain what is missing."""
    key = os.environ.get("PINECONE_PROD_KEY")
    if key:
        return key
    server_env = Path("../../server/.env")
    key = dotenv_values(server_env).get("PINECONE_KEY") if server_env.exists() else None
    if key:
        return key
    raise RuntimeError(
        "No production Pinecone key. Set PINECONE_PROD_KEY, or check out the server "
        f"repo so that {server_env} exists with PINECONE_KEY in it. Do not substitute "
        "models/.env PINECONE_API_KEY: that is a dev key for an empty project, and it "
        "fails by returning no matches rather than by raising."
    )


def connect_to_index(host):
    """Open a read-only connection to the production conference index."""
    # The client stays local to this function, so the notebook never holds a handle
    # that can write.
    pc = Pinecone(api_key=get_pinecone_prod_key())
    return ReadOnlyIndex(pc.Index(host=host))

In [ ]:
index = connect_to_index(index_host)
index.describe_index_stats()

## Step 1 — embed the question

An embedding model turns text into a list of numbers positioned so that texts about
the same thing land near each other. Every chunk of every talk in the index went
through this exact model when it was indexed; now the question goes through it too,
which is what makes the two comparable.

In [ ]:
def embed_question(question):
    """Return the 1536-float embedding of a question."""
    response = openai_client.embeddings.create(
        input=[question], model=text_embedding_model_name
    )
    return response.data[0].embedding


embedding = embed_question("What are the blessings of keeping the sabbath day holy?")
len(embedding), embedding[:5]

## Step 2 — query the index

Pinecone compares the question vector against every indexed chunk by cosine
similarity and hands back the closest `top_k`. Each match carries the metadata that
was stored alongside it — `year`, `month`, `url`, `title`, `author`, `author_role`,
`anchor`, and the chunk's own `text` — which is what makes it possible to both quote
the passage and cite it.

Scores are only comparable within one response. There is no fixed number above which
a match is "good", so this does not filter on one.

The live server reaches the same index through the gRPC client
(`pinecone.grpc.PineconeGRPC`) for lower latency. The REST client used here talks to
the same data and takes the same arguments.

In [ ]:
def search(question, top_k=search_limit):
    """Return the indexed passages nearest to a question, nearest first."""
    response = index.query(
        vector=embed_question(question), top_k=top_k, include_metadata=True
    )
    return response.matches


matches = search("What are the blessings of keeping the sabbath day holy?")
for match in matches[:5]:
    print(f"{match.score:.4f}  {match.metadata['title']} ({match.metadata['author']})")

## Step 3 — build a prompt from the retrieved passages

This is the "augmented" half of RAG, and it is just string concatenation: an
instruction, the retrieved passages, then the question.

The one subtlety is the budget. Twenty passages of up to 2000 characters each would
be a 40,000-character prompt — slow, expensive, and mostly padding, since the
passages that matter are at the top. So the loop adds passages in rank order and
stops before the prompt would exceed `prompt_limit` characters. That is exactly what
`reduce_k_below_max_tokens` was doing inside the old chain, and it is four lines.

A `PromptTemplate` was never more than `str.format` either, which the next section
makes use of.

In [ ]:
role_content = "You are an apostle of the Church of Jesus Christ of Latter-day Saints"
prompt_content = (
    "Answer the question as truthfully as possible using the provided context, "
    'and if answer is not contained within the text below, say "Sorry, I don\'t know".'
)


def build_prompt(question, contexts, limit=prompt_limit):
    """Return a prompt holding as many contexts as fit in limit characters."""

    def prompt_for(ctxs):
        return (
            prompt_content
            + "\n\nContext:\n"
            + "\n\n---\n\n".join(ctxs)
            + f"\n\nQuestion: {question}\nAnswer:"
        )

    n_contexts = 0
    while n_contexts < len(contexts) and len(prompt_for(contexts[: n_contexts + 1])) < limit:
        n_contexts += 1
    return prompt_for(contexts[:n_contexts]), n_contexts


prompt, n_contexts = build_prompt(
    "What are the blessings of keeping the sabbath day holy?",
    [match.metadata["text"] for match in matches],
)
print(f"{n_contexts} of {len(matches)} passages fit in {len(prompt)} characters\n")
print(prompt[:1500], "...")

## Step 4 — ask the model

The retrieved passages go in as the user turn; the persona goes in as the system
turn. Nothing else about the conference is in the model's context, so an answer that
is not supported by the passages above is the model making something up — which is
what the "Sorry, I don't know" instruction is there to discourage.

In [ ]:
def ask(prompt, role=role_content):
    """Ask the generation model a prompt and return its answer."""
    response = openai_client.responses.create(
        model=text_generation_model_name,
        input=[
            {"role": "system", "content": role},
            {"role": "user", "content": prompt},
        ],
        reasoning={"effort": "low"},
        max_output_tokens=max_answer_tokens,
    )
    return response.output_text.strip()

## Answer a question

The four steps, end to end.

In [ ]:
def answer_question(question, top_k=search_limit, limit=prompt_limit):
    """Answer a question from the conference index, returning the answer and sources."""
    matches = search(question, top_k=top_k)
    prompt, n_contexts = build_prompt(
        question, [match.metadata["text"] for match in matches], limit=limit
    )
    return ask(prompt), matches[:n_contexts]

In [ ]:
query = "What are the blessings of keeping the sabbath day holy?"

answer, sources = answer_question(query)
print(textwrap.fill(answer, width=88))

## Show the sources

The old `RetrievalQAWithSourcesChain` returned sources because an answer you cannot
check is not much of an answer. Here the sources are simply the matches that made it
into the prompt — the ones `build_prompt` had room for — so nothing is cited that the
model could not have read.

The `anchor` metadata is what lets a link land on the right paragraph of a talk
rather than at the top of it; the live site builds its deep links the same way.

In [ ]:
def show_sources(matches):
    """Print the talks behind an answer, with deep links into the paragraph."""
    for i, match in enumerate(matches, start=1):
        meta = match.metadata
        url = f"{meta['url']}#{meta['anchor']}" if meta.get("anchor") else meta["url"]
        print(f"{i}. {meta['title']} — {meta['author']}, {meta['month']}/{meta['year']}")
        print(f"   {url}")
        print(f"   {textwrap.shorten(meta['text'], width=160, placeholder=' ...')}\n")


show_sources(sources)

## Change the prompt

A different instruction gives a different answer over the very same passages. This is
the template the notebook used to carry — `{summaries}` and `{question}` are the
placeholders the old `PromptTemplate` filled in, and `str.format` fills them in just
as well.

In [ ]:
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum. Keep the answer as concise as possible.
Always say "thanks for asking!" at the end of the answer.
{summaries}
Question: {question}
Helpful Answer:"""

matches = search(query)
summaries = "\n\n---\n\n".join(match.metadata["text"] for match in matches[:5])

print(textwrap.fill(ask(template.format(summaries=summaries, question=query)), width=88))

## Try other questions

Retrieval is the half that fails quietly: when an answer disappoints, print the
matches first and see whether the right passages were ever found. If they were not,
no amount of prompt wording will rescue the answer.

In [ ]:
questions = [
    "What are the blessings of keeping the sabbath day holy?",
    "How can I receive personal revelation?",
    "What did they teach about ministering to others?",
]

for question in questions:
    answer, sources = answer_question(question)
    print(f"Q: {question}")
    print(textwrap.fill(answer, width=88, initial_indent="A: ", subsequent_indent="   "))
    print(f"   [{len(sources)} sources: {', '.join(s.metadata['title'] for s in sources[:3])} ...]\n")